# Coral Reef Condition Estimation - Training Notebook
**U-Net + EfficientNet-B3 | Semantic Segmentation**

Notebook ini untuk training model segmentasi terumbu karang di Google Colab dengan GPU.

### Langkah:
1. Aktifkan GPU: `Runtime > Change runtime type > T4 GPU`
2. Jalankan semua cell dari atas ke bawah
3. Download file model `.pth` yang sudah dilatih

## 1. Install Dependencies & Cek GPU

In [ ]:
!pip install -q segmentation-models-pytorch albumentations roboflow

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: GPU tidak aktif! Aktifkan di Runtime > Change runtime type > T4 GPU')

## 2. Download Dataset dari Roboflow
Ganti `API_KEY`, `WORKSPACE`, `PROJECT`, dan `VERSION` sesuai project Roboflow Anda.

In [ ]:
from roboflow import Roboflow

# ============================================
# GANTI DENGAN KREDENSIAL ROBOFLOW ANDA
# ============================================
API_KEY = "YOUR_ROBOFLOW_API_KEY"  # <-- Ganti ini!
WORKSPACE = "your-workspace"       # <-- Ganti ini!
PROJECT = "coral-reef-segmentation" # <-- Ganti ini!
VERSION = 1                         # <-- Ganti ini!
# ============================================

rf = Roboflow(api_key=API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
version = project.version(VERSION)
dataset = version.download('coco-segmentation', location='/content/dataset')
print(f'Dataset downloaded to: {dataset.location}')

**Alternatif:** Jika sudah punya ZIP dataset, upload manual lalu extract:

In [ ]:
# === ALTERNATIF: Upload ZIP dari komputer lokal ===
# Uncomment baris di bawah jika ingin upload manual

# from google.colab import files
# uploaded = files.upload()  # Pilih file ZIP Anda
# !mkdir -p /content/dataset
# !unzip -q -o *.zip -d /content/dataset
# !ls /content/dataset

## 3. Konfigurasi

In [ ]:
from pathlib import Path

# ── Konfigurasi Training ──
DATASET_DIR = Path('/content/dataset')
MODEL_SAVE_PATH = Path('/content/best_unet_efficientnet.pth')

IMG_SIZE = 256
NUM_CLASSES = 1
BATCH_SIZE = 16          # Lebih besar karena GPU punya VRAM
LEARNING_RATE = 1e-4
NUM_EPOCHS = 50          # Full training dengan GPU
ENCODER_NAME = 'efficientnet-b3'
ENCODER_WEIGHTS = 'imagenet'

# ImageNet normalization
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## 4. Dataset & DataLoader

In [ ]:
import json
import cv2
import numpy as np
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2


class CoralReefDataset(Dataset):
    def __init__(self, root_dir, img_size=256, mean=[0.485,0.456,0.406],
                 std=[0.229,0.224,0.225], augment=False):
        self.root_dir = Path(root_dir)
        self.img_size = img_size

        # Load COCO annotations
        with open(self.root_dir / '_annotations.coco.json', 'r') as f:
            self.coco = json.load(f)

        self.images = {img['id']: img for img in self.coco['images']}
        self.img_to_anns = {}
        for ann in self.coco['annotations']:
            self.img_to_anns.setdefault(ann['image_id'], []).append(ann)
        self.image_ids = list(self.img_to_anns.keys())

        # Augmentation pipeline
        if augment:
            self.transform = A.Compose([
                A.Resize(img_size, img_size),
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.3),
                A.Affine(shift_limit=0.1, scale_limit=0.2, rotate_limit=30,
                         border_mode=cv2.BORDER_CONSTANT, p=0.5),
                A.OneOf([
                    A.GaussianBlur(blur_limit=(3, 7), p=1.0),
                    A.MotionBlur(blur_limit=7, p=1.0),
                ], p=0.3),
                A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.3, hue=0.1, p=0.4),
                A.GaussNoise(p=0.2),
                A.Normalize(mean=mean, std=std),
                ToTensorV2(),
            ])
        else:
            self.transform = A.Compose([
                A.Resize(img_size, img_size),
                A.Normalize(mean=mean, std=std),
                ToTensorV2(),
            ])

    def _polygon_to_mask(self, annotations, height, width):
        mask = np.zeros((height, width), dtype=np.uint8)
        for ann in annotations:
            for seg in ann.get('segmentation', []):
                pts = np.array(seg, dtype=np.float32).reshape(-1, 2).astype(np.int32)
                cv2.fillPoly(mask, [pts], color=1)
        return mask

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img_info = self.images[img_id]

        image = cv2.imread(str(self.root_dir / img_info['file_name']))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        h, w = img_info['height'], img_info['width']
        mask = self._polygon_to_mask(self.img_to_anns.get(img_id, []), h, w)

        transformed = self.transform(image=image, mask=mask)
        image_t = transformed['image']
        mask_t = transformed['mask'].unsqueeze(0).float()

        return image_t, mask_t


# Buat DataLoader
train_ds = CoralReefDataset(DATASET_DIR / 'train', IMG_SIZE, MEAN, STD, augment=True)
val_ds = CoralReefDataset(DATASET_DIR / 'valid', IMG_SIZE, MEAN, STD, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds)} samples -> {len(train_loader)} batches')
print(f'Valid: {len(val_ds)} samples -> {len(val_loader)} batches')

# Quick test
img, msk = train_ds[0]
print(f'Image: {img.shape}, Mask: {msk.shape}, Mask values: {msk.unique().tolist()}')

## 5. Model U-Net + EfficientNet

In [ ]:
import segmentation_models_pytorch as smp

model = smp.Unet(
    encoder_name=ENCODER_NAME,
    encoder_weights=ENCODER_WEIGHTS,
    in_channels=3,
    classes=NUM_CLASSES,
    activation=None,  # Raw logits, sigmoid diterapkan di loss
)
model = model.to(DEVICE)

# Hitung total parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

## 6. Metrics

In [ ]:
def calculate_metrics(pred, true, threshold=0.5):
    pred_binary = (torch.sigmoid(pred.detach()) > threshold).float().view(-1)
    true_binary = true.float().view(-1)

    tp = (pred_binary * true_binary).sum().item()
    fp = (pred_binary * (1 - true_binary)).sum().item()
    fn = ((1 - pred_binary) * true_binary).sum().item()
    tn = ((1 - pred_binary) * (1 - true_binary)).sum().item()

    total = tp + fp + fn + tn
    acc = (tp + tn) / total if total > 0 else 0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    iou = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0

    return {'accuracy': round(acc,4), 'precision': round(prec,4),
            'recall': round(rec,4), 'f1_score': round(f1,4), 'iou': round(iou,4)}

## 7. Training Loop

In [ ]:
import torch.nn as nn
from tqdm.auto import tqdm

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

# History untuk plotting
history = {'train_loss': [], 'val_loss': [], 'train_f1': [], 'val_f1': [], 'train_iou': [], 'val_iou': []}
best_val_loss = float('inf')

for epoch in range(1, NUM_EPOCHS + 1):
    # === TRAINING ===
    model.train()
    train_loss = 0
    train_metrics = {'accuracy':0, 'precision':0, 'recall':0, 'f1_score':0, 'iou':0}

    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{NUM_EPOCHS} [Train]')
    for images, masks in pbar:
        images, masks = images.to(DEVICE), masks.to(DEVICE)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        batch_met = calculate_metrics(logits, masks)
        for k in train_metrics:
            train_metrics[k] += batch_met[k]
        pbar.set_postfix(loss=f'{loss.item():.4f}')

    n_train = len(train_loader)
    train_loss /= n_train
    train_metrics = {k: round(v/n_train, 4) for k, v in train_metrics.items()}

    # === VALIDATION ===
    model.eval()
    val_loss = 0
    val_metrics = {'accuracy':0, 'precision':0, 'recall':0, 'f1_score':0, 'iou':0}

    with torch.no_grad():
        for images, masks in tqdm(val_loader, desc=f'Epoch {epoch}/{NUM_EPOCHS} [Valid]'):
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            logits = model(images)
            loss = criterion(logits, masks)
            val_loss += loss.item()
            batch_met = calculate_metrics(logits, masks)
            for k in val_metrics:
                val_metrics[k] += batch_met[k]

    n_val = len(val_loader)
    val_loss /= n_val
    val_metrics = {k: round(v/n_val, 4) for k, v in val_metrics.items()}

    scheduler.step(val_loss)

    # Save history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_f1'].append(train_metrics['f1_score'])
    history['val_f1'].append(val_metrics['f1_score'])
    history['train_iou'].append(train_metrics['iou'])
    history['val_iou'].append(val_metrics['iou'])

    # Print results
    print(f'\nEpoch {epoch}/{NUM_EPOCHS}')
    print(f'  Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}')
    print(f'  Train -> Acc: {train_metrics["accuracy"]} | F1: {train_metrics["f1_score"]} | IoU: {train_metrics["iou"]}')
    print(f'  Valid -> Acc: {val_metrics["accuracy"]} | F1: {val_metrics["f1_score"]} | IoU: {val_metrics["iou"]}')

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), str(MODEL_SAVE_PATH))
        print(f'  >>> Best model saved! (val_loss: {val_loss:.4f})')

print(f'\nTraining selesai! Best val_loss: {best_val_loss:.4f}')

## 8. Visualisasi Training History

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['val_loss'], label='Val Loss')
axes[0].set_title('Loss')
axes[0].legend()
axes[0].set_xlabel('Epoch')

# F1 Score
axes[1].plot(history['train_f1'], label='Train F1')
axes[1].plot(history['val_f1'], label='Val F1')
axes[1].set_title('F1 Score')
axes[1].legend()
axes[1].set_xlabel('Epoch')

# IoU
axes[2].plot(history['train_iou'], label='Train IoU')
axes[2].plot(history['val_iou'], label='Val IoU')
axes[2].set_title('IoU')
axes[2].legend()
axes[2].set_xlabel('Epoch')

plt.tight_layout()
plt.savefig('/content/training_history.png', dpi=150)
plt.show()

## 9. Test Prediksi (Sample Visualisasi)

In [ ]:
# Load best model
model.load_state_dict(torch.load(str(MODEL_SAVE_PATH), map_location=DEVICE, weights_only=True))
model.eval()

# Ambil 4 sampel dari validation set
fig, axes = plt.subplots(4, 3, figsize=(15, 20))

for i in range(4):
    img_t, mask_gt = val_ds[i]

    with torch.no_grad():
        pred = model(img_t.unsqueeze(0).to(DEVICE))
        pred_mask = (torch.sigmoid(pred) > 0.5).float().squeeze().cpu().numpy()

    # Denormalize image
    img_np = img_t.permute(1, 2, 0).numpy()
    img_np = img_np * np.array(STD) + np.array(MEAN)
    img_np = np.clip(img_np, 0, 1)

    axes[i, 0].imshow(img_np)
    axes[i, 0].set_title('Original')
    axes[i, 0].axis('off')

    axes[i, 1].imshow(mask_gt.squeeze(), cmap='Reds')
    axes[i, 1].set_title('Ground Truth')
    axes[i, 1].axis('off')

    axes[i, 2].imshow(pred_mask, cmap='Reds')
    axes[i, 2].set_title('Prediction')
    axes[i, 2].axis('off')

plt.suptitle('Validasi: Original | Ground Truth | Prediction', fontsize=16)
plt.tight_layout()
plt.savefig('/content/sample_predictions.png', dpi=150)
plt.show()

## 10. Download Model
Jalankan cell ini untuk download file model ke komputer lokal.

Setelah download, letakkan file di:
```
backend/model/weights/best_unet_efficientnet.pth
```

In [ ]:
from google.colab import files

# Download model weights
files.download(str(MODEL_SAVE_PATH))
print('Model berhasil di-download!')
print('Letakkan file di: backend/model/weights/best_unet_efficientnet.pth')

In [ ]:
# Download juga grafik training history
files.download('/content/training_history.png')
files.download('/content/sample_predictions.png')